# ALiBi: Attention with Linear Biases for Length Extrapolation Beyond Training

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/research_4)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q torch transformers peft bitsandbytes

In [ ]:
import torch
import math

def compute_alibi_slopes(num_heads: int) -> torch.Tensor:
    """
    Compute ALiBi slopes for each attention head.
    Returns tensor of shape (num_heads,).
    """
    # Start with n nearest power of 2 >= num_heads
    n = 2 ** math.ceil(math.log2(num_heads))

    # Geometric series: 2^(-8/n), 2^(-16/n), ..., 2^(-8)
    slopes = torch.tensor([2 ** (-8 * h / n) for h in range(1, n + 1)])

    # If num_heads is not a power of 2, take alternating values for extra heads
    if n != num_heads:
        extra = compute_alibi_slopes(n // 2)
        # Interleave main and extra slopes
        slopes = torch.stack([slopes[i//2] if i % 2 == 0 else extra[i//2]
                               for i in range(num_heads)])

    return slopes[:num_heads]

slopes = compute_alibi_slopes(8)
print("ALiBi slopes for 8 heads:")
for i, s in enumerate(slopes):
    print(f"  Head {i+1}: {s:.6f} (= 2^{math.log2(s.item()):.2f})")

In [ ]:
import torch

def build_alibi_bias(seq_len: int, slopes: torch.Tensor) -> torch.Tensor:
    """
    Build ALiBi bias matrix for all heads.
    Returns tensor of shape (num_heads, seq_len, seq_len).
    """
    # Distance matrix: entry (i, j) = j - i (causal: only j <= i matters)
    positions = torch.arange(seq_len)
    distance = positions.unsqueeze(0) - positions.unsqueeze(1)  # (seq_len, seq_len)

    # For causal attention, only past positions matter: distance <= 0
    # ALiBi uses |i - j| which equals -distance for j <= i
    alibi = -torch.abs(distance).float()  # (seq_len, seq_len)

    # Scale by each head's slope: (num_heads, 1, 1) * (1, seq_len, seq_len)
    alibi = alibi.unsqueeze(0) * slopes.view(-1, 1, 1)  # (num_heads, seq_len, seq_len)

    return alibi

seq_len = 8
slopes = torch.tensor([0.5, 0.125])  # 2 heads for demo

bias = build_alibi_bias(seq_len, slopes)
print(f"ALiBi bias shape: {bias.shape}")
print(f"\nHead 0 (slope=0.5) — strong local bias:")
print(bias[0].to(torch.int).numpy())
print(f"\nHead 1 (slope=0.125) — weaker local bias:")
print(bias[1].round(decimals=3).numpy())

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def compute_alibi_slopes(num_heads: int) -> torch.Tensor:
    n = 2 ** math.ceil(math.log2(num_heads))
    slopes = torch.tensor([2 ** (-8 * h / n) for h in range(1, n + 1)])
    if n != num_heads:
        extra = torch.tensor([2 ** (-8 * h / (n // 2)) for h in range(1, n // 2 + 1)])
        result = []
        for i in range(num_heads):
            result.append(slopes[i // 2] if i % 2 == 0 else extra[i // 2])
        slopes = torch.stack(result)
    return slopes[:num_heads]

def build_alibi_bias(seq_len: int, slopes: torch.Tensor) -> torch.Tensor:
    positions = torch.arange(seq_len)
    distance = positions.unsqueeze(0) - positions.unsqueeze(1)
    alibi = -torch.abs(distance).float().unsqueeze(0) * slopes.view(-1, 1, 1)
    return alibi

class ALiBiAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = math.sqrt(self.head_dim)

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

        # Precompute slopes — not learned parameters
        slopes = compute_alibi_slopes(num_heads)
        self.register_buffer('slopes', slopes)

    def forward(self, x: torch.Tensor, causal: bool = True) -> torch.Tensor:
        B, T, C = x.shape

        # Project to Q, K, V
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)  # each: (B, H, T, head_dim)

        # Attention scores (no positional encoding in Q/K — that's the ALiBi point)
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale  # (B, H, T, T)

        # Add ALiBi bias
        alibi = build_alibi_bias(T, self.slopes).to(x.device)  # (H, T, T)
        scores = scores + alibi.unsqueeze(0)  # broadcast over batch

        # Causal mask
        if causal:
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)  # (B, H, T, head_dim)

        # Reshape and project
        out = out.transpose(1, 2).contiguous().reshape(B, T, C)
        return self.out(out)

# Test
torch.manual_seed(42)
model = ALiBiAttention(d_model=256, num_heads=8)
x = torch.randn(2, 64, 256)  # batch=2, seq_len=64, d_model=256

out = model(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,} (no position embedding params)")
print(f"Slopes: {model.slopes.tolist()[:4]}...")

In [ ]:
import torch
import math

def compute_alibi_slopes(num_heads):
    n = 2 ** math.ceil(math.log2(num_heads))
    slopes = torch.tensor([2 ** (-8 * h / n) for h in range(1, n + 1)])
    return slopes[:num_heads]

def build_alibi_bias(seq_len, slopes):
    positions = torch.arange(seq_len)
    distance = positions.unsqueeze(0) - positions.unsqueeze(1)
    return -torch.abs(distance).float().unsqueeze(0) * slopes.view(-1, 1, 1)

slopes = compute_alibi_slopes(8)

# Compare attention patterns at training length vs 2x length
for train_len, test_len in [(64, 64), (64, 128)]:
    bias = build_alibi_bias(test_len, slopes)
    # Simulate uniform attention logits (no actual content)
    logits = torch.zeros(1, 8, test_len, test_len)
    logits = logits + bias.unsqueeze(0)

    # Apply causal mask
    mask = torch.triu(torch.ones(test_len, test_len), diagonal=1).bool()
    logits = logits.masked_fill(mask, float('-inf'))

    attn = torch.softmax(logits, dim=-1)

    # For the last token, what fraction of attention is on last 64 tokens?
    last_token_attn = attn[0, :, -1, :]  # (num_heads, test_len)
    recent_fraction = last_token_attn[:, -train_len:].sum(dim=-1)

    print(f"Train={train_len}, Test={test_len}: "
          f"fraction on last {train_len} tokens per head:")
    print(f"  {[f'{v:.2%}' for v in recent_fraction.tolist()]}")

In [ ]:
from transformers import AutoConfig

# MPT-7B uses ALiBi
config = AutoConfig.from_pretrained("mosaicml/mpt-7b")
print(f"Model type: {config.model_type}")
print(f"ALiBi: {getattr(config, 'alibi', False)}")
print(f"Max sequence length: {config.max_seq_len}")
print(f"Attention config: {config.attn_config}")